## 🧩 第一步：认识 Tensor —— PyTorch 的“灵魂”

在 PyTorch 里，几乎所有东西（数据、参数、输入、输出）都是 Tensor（张量）

👉 它是 多维数组，类似于 NumPy 的 ndarray，但可以在 GPU 上加速运算。


In [2]:
import torch

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("x=\n", x)

print("shape:", x.shape)
print("dtype:", x.dtype)
print("device", x.device)

a = torch.zeros((2, 3))
b = torch.ones((2, 3))
c = torch.randn((2, 3))
print("a = ", a)
print("b = ", b)
print("c = ", c)

x=
 tensor([[1., 2.],
        [3., 4.]])
shape: torch.Size([2, 2])
dtype: torch.float32
device cpu
a =  tensor([[0., 0., 0.],
        [0., 0., 0.]])
b =  tensor([[1., 1., 1.],
        [1., 1., 1.]])
c =  tensor([[ 1.5550, -0.5703, -1.3767],
        [ 0.8777,  1.0235, -0.6726]])


## 🧩 第二步：张量的基本运算（Tensor Operations）

我们主要学习以下几个方面：

1. 张量的形状操作（如 reshape、view、unsqueeze、squeeze）

2. 张量的数学运算（加减乘除、矩阵乘法）

3. 广播机制（Broadcasting）

4. 索引与切片

### 1. 张量的形状操作（如 reshape、view、unsqueeze、squeeze）

#### 🧠一、reshape 与 view 的区别


| 函数          | 功能               | 底层机制 | 是否创建新内存    |
| ----------- | ---------------- | ---- | ---------- |
| `reshape()` | 改变形状，自动选择是否创建新内存 | 智能   | 有时创建新张量    |
| `view()`    | 改变形状，但要求内存连续     | 不智能  | 必须保证张量是连续的 |


个人总结：
在 PyTorch（乃至 NumPy）中，张量的元素通常在内存中是连续排列的。
但某些操作（比如 .t() 转置、切片）会改变访问顺序，使它不再连续。

view(): 

要求张量必须是连续的。
因为 view() 只是重新解释内存布局，它不会复制数据。
如果张量不是连续的，它就无法直接“重新解释”，因此会报错。

reshape() 会尝试：
1. 如果可能，不复制数据（直接用 view）；
2. 如果不行，就自动复制一份新的连续内存。

一般而言就是转置过的张量无法view，不追求性能就用万能的reshape


#### 🧠二、unsqueeze 和 squeeze


##### 🧪unsqueeze

x = torch.tensor([1, 2, 3])  # shape: (3,)

那么只能

x.unsqueeze(0)  # 在第0个维度插入 → shape: (1, 3)

x.unsqueeze(1)  # 在第1个维度插入 → shape: (3, 1)

看shape就好，参数为几就表示在shape下标为几的地方加一个维度

In [25]:
x = torch.tensor([1, 2, 3])
print(x.shape)
x.unsqueeze(0)
print(x.shape)

torch.Size([3])
torch.Size([3])


为什么还没变呢？

因为unsqueeze函数不会在原地修改，而是将修改结果作为返回值返回

下面这样就好了

In [26]:
x = torch.tensor([1, 2, 3])
print(x.shape)
x = x.unsqueeze(0)
print(x.shape)

torch.Size([3])
torch.Size([1, 3])


✅ PyTorch 约定俗成：
带下划线 _ 结尾的函数（如 .add_()、 .zero_()、 .unsqueeze_()）表示“原地修改”。

##### 🧪squeeze

squeeze(dim)

删除大小为 1 的维度。

In [29]:
x = torch.randn(1, 3, 1, 5)
x.shape   # torch.Size([1, 3, 1, 5])

print(x.squeeze().shape)        # torch.Size([3, 5]) → 删除所有为1的维度
print(x.squeeze(0).shape)       # torch.Size([3, 1, 5]) → 仅删除第0个维度
print(x.squeeze(1).shape)       # torch.Size([1, 3, 1, 5]) 大小不为1的维度删不掉！！！

torch.Size([3, 5])
torch.Size([3, 1, 5])
torch.Size([1, 3, 1, 5])


🌰 举个实际例子

在神经网络中，模型输入通常需要形状 [batch_size, channel, height, width]。
如果你只有一张灰度图（没有 batch 维度），就可以：

In [32]:
img = torch.randn(1, 28, 28)
img = img.unsqueeze(0)  # shape: (1, 1, 28, 28)
img.shape


torch.Size([1, 1, 28, 28])